# 🧱 Lab: Document Data Extractor

**Module 3: Prompt Engineering** | **Duration: ~1 hour** | **Type: Wall Lab**

---

## Learning Objectives

By the end of this lab, you will be able to:

1. **Extract** structured data from unstructured text using LLM prompts
2. **Compare** zero-shot vs few-shot extraction approaches
3. **Apply** JSON mode for guaranteed valid output
4. **Validate** extracted data with Pydantic models

## Concepts Covered

| Concept | From |
|---------|------|
| Structured Output | mini-structured |
| Few-Shot Prompting | mini-few-shot |
| JSON Mode | mini-structured |
| Output Parsing | This lab |

## Prerequisites

- **mini-structured**: Structured output basics and JSON mode
- **mini-few-shot**: Few-shot prompting patterns
- OpenAI API key configured in `.env`

In [18]:
import json
import os
from pathlib import Path
from openai import OpenAI
from pydantic import BaseModel
from typing import List, Optional
from dotenv import load_dotenv
from IPython.display import Markdown, display

def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))

load_dotenv()
client = OpenAI()

# Load all invoice files from data folder
DATA_DIR = Path('../data/invoices/')
INVOICES = {}

for file in sorted(DATA_DIR.glob('invoice_*.txt')):
    INVOICES[file.stem] = file.read_text(encoding='utf-8')

# Use second invoice as sample for initial examples
SAMPLE_INVOICE = list(INVOICES.values())[1] if INVOICES else ""

print(f'✓ Loaded {len(INVOICES)} invoices from data folder:')
for name in INVOICES.keys():
    print(f'  - {name}')

print(f'\n📄 Sample invoice preview (second one):')
print(SAMPLE_INVOICE[:300] + '...' if len(SAMPLE_INVOICE) > 500 else SAMPLE_INVOICE)

✓ Loaded 6 invoices from data folder:
  - invoice_001_corporate
  - invoice_002_simple_receipt
  - invoice_003_email_style
  - invoice_004_european_format
  - invoice_005_handwritten_style
  - invoice_006_minimal

📄 Sample invoice preview (second one):
Invoice

CloudHost Services
------------------
Invoice #: REC-8847
Date: March 22, 2024

Customer: DataFlow Analytics

Services Rendered:
* Cloud Hosting (Monthly)     $299.00
* Extra Storage 500GB         $45.00
* SSL Certificate             $12.00
* Priority Support            $50.00

------------------------
TOTAL PAID: $406.00
------------------------

Payment Method: Credit Card (****4521)
Transaction ID: TXN-20240322-8847

Questions? support@cloudhost.io



## 2. Zero-Shot Extraction

Ask the LLM to extract data without any examples.

In [19]:
def zero_shot_extract(doc, fields):
    """Extract fields from document using zero-shot prompting."""
    prompt = f'''Extract the following fields from the document: {fields}

Document:
{doc}

Return the extracted data as JSON.'''
    
    resp = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0
    )
    return resp.choices[0].message.content

# Test zero-shot extraction
result = zero_shot_extract(SAMPLE_INVOICE, ['invoice_number', 'vendor', 'total'])
print('Zero-shot extraction result:')
print(result)

Zero-shot extraction result:
```json
{
  "invoice_number": "REC-8847",
  "vendor": "CloudHost Services",
  "total": "$406.00"
}
```


## 3. Few-Shot Extraction

Provide examples to guide the extraction format.

In [20]:
def few_shot_extract(doc, examples):
    """Extract data using few-shot examples."""
    # Format examples
    examples_text = '\n\n'.join([
        f'Document: {ex["doc"]}\nExtracted: {json.dumps(ex["result"])}'
        for ex in examples
    ])
    
    prompt = f'''Extract structured data from documents.

Examples:
{examples_text}

Now extract from this document:
{doc}

Extracted:'''
    
    resp = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0
    )
    return resp.choices[0].message.content

# Define few-shot examples
examples = [
    {
        'doc': 'Invoice #A1 from CompanyX to CustomerY, Total $100',
        'result': {'invoice_number': 'A1', 'vendor': 'CompanyX', 'customer': 'CustomerY', 'total': 100}
    },
    {
        'doc': 'INV-99 Seller: MegaCorp Buyer: SmallBiz Amount: $500',
        'result': {'invoice_number': 'INV-99', 'vendor': 'MegaCorp', 'customer': 'SmallBiz', 'total': 500}
    }
]

result = few_shot_extract(SAMPLE_INVOICE, examples)
print('Few-shot extraction result:')
print(result)

Few-shot extraction result:
{"invoice_number": "REC-8847", "vendor": "CloudHost Services", "customer": "DataFlow Analytics", "total": 406}


## 4. JSON Mode

Force the LLM to return valid JSON.

In [21]:
def json_extract(doc, schema_description):
    """Extract data with guaranteed JSON output."""
    prompt = f'''Extract data from the document according to this schema: {schema_description}

Document:
{doc}

Return only valid JSON with the extracted fields.'''
    
    resp = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{'role': 'user', 'content': prompt}],
        response_format={'type': 'json_object'},
        temperature=0
    )
    return json.loads(resp.choices[0].message.content)

# Test JSON mode
result = json_extract(SAMPLE_INVOICE, 'invoice_number (string), vendor (string), customer (string), total (number)')
print('JSON mode extraction result:')
print(json.dumps(result, indent=2))
print(f'\nType of result: {type(result)}')

JSON mode extraction result:
{
  "invoice_number": "REC-8847",
  "vendor": "CloudHost Services",
  "customer": "DataFlow Analytics",
  "total": 406.0
}

Type of result: <class 'dict'>


## 5. Pydantic Validation

Use Pydantic models for type validation and data integrity.

In [22]:
# Define Pydantic model for invoice
class Invoice(BaseModel):
    invoice_number: str
    vendor: str
    customer: str
    total: float
    date: Optional[str] = None

def extract_pydantic(doc, model):
    """Extract and validate data using Pydantic model."""
    schema = model.model_json_schema()
    
    prompt = f'''Extract data from the document according to this JSON schema:
{json.dumps(schema, indent=2)}

Document:
{doc}

Return only valid JSON matching the schema.'''
    
    resp = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{'role': 'user', 'content': prompt}],
        response_format={'type': 'json_object'},
        temperature=0
    )
    
    # Parse and validate with Pydantic
    data = json.loads(resp.choices[0].message.content)
    return model(**data)

# Extract and validate
invoice = extract_pydantic(SAMPLE_INVOICE, Invoice)
print('Pydantic validated result:')
print(f'  Invoice Number: {invoice.invoice_number}')
print(f'  Vendor: {invoice.vendor}')
print(f'  Customer: {invoice.customer}')
print(f'  Total: ${invoice.total}')
print(f'  Date: {invoice.date}')

Pydantic validated result:
  Invoice Number: REC-8847
  Vendor: CloudHost Services
  Customer: DataFlow Analytics
  Total: $406.0
  Date: March 22, 2024


## 6. Building a Document Processor

Combine everything into a reusable processor that handles **multiple invoice formats**. 

The key insight: by using a well-defined schema and clear prompts, the LLM can extract the same structured data from vastly different document formats!

In [23]:
class DocumentProcessor:
    """A reusable document data extractor."""
    
    def __init__(self, client, model='gpt-4o-mini'):
        self.client = client
        self.model = model
    
    def extract(self, doc, pydantic_model):
        """Extract structured data from a document."""
        schema = pydantic_model.model_json_schema()
        
        prompt = f'''You are a document data extraction assistant.
Extract all relevant fields from the document according to this schema:
{json.dumps(schema, indent=2)}

Document:
{doc}

Return only valid JSON. Use null for missing fields.'''
        
        resp = self.client.chat.completions.create(
            model=self.model,
            messages=[{'role': 'user', 'content': prompt}],
            response_format={'type': 'json_object'},
            temperature=0
        )
        
        data = json.loads(resp.choices[0].message.content)
        return pydantic_model(**data)

# Create processor
processor = DocumentProcessor(client)

# Process ALL invoices from different formats
md("## 📊 Processing All Invoices\n\nExtracting structured data from **6 different invoice formats**...\n\n---")

results = []
for name, content in INVOICES.items():
    try:
        invoice = processor.extract(content, Invoice)
        results.append({
            'source': name,
            'invoice_number': invoice.invoice_number,
            'vendor': invoice.vendor,
            'customer': invoice.customer,
            'total': invoice.total,
            'date': invoice.date,
            'status': '✅'
        })
    except Exception as e:
        results.append({
            'source': name,
            'invoice_number': 'ERROR',
            'vendor': str(e)[:30],
            'customer': '-',
            'total': 0,
            'date': '-',
            'status': '❌'
        })

# Display results as markdown table
table = "| Status | Source File | Invoice # | Vendor | Customer | Total | Date |\n"
table += "|:------:|-------------|-----------|--------|----------|------:|------|\n"

for r in results:
    total_str = f"${r['total']:,.2f}" if isinstance(r['total'], (int, float)) else str(r['total'])
    table += f"| {r['status']} | {r['source'][:25]} | {r['invoice_number']} | {r['vendor'][:20]} | {r['customer'][:15]} | {total_str} | {r['date']} |\n"

md(table)

# Summary
total_extracted = sum(r['total'] for r in results if r['status'] == '✅')
md(f"\n---\n\n### 💰 Summary\n\n- **Invoices processed:** {len(results)}\n- **Successful extractions:** {sum(1 for r in results if r['status'] == '✅')}\n- **Total value extracted:** ${total_extracted:,.2f}")

## 📊 Processing All Invoices

Extracting structured data from **6 different invoice formats**...

---

| Status | Source File | Invoice # | Vendor | Customer | Total | Date |
|:------:|-------------|-----------|--------|----------|------:|------|
| ✅ | invoice_001_corporate | INV-2024-0842 | TechCorp Solutions I | Acme Corporatio | $24,686.46 | 2024-01-15 |
| ✅ | invoice_002_simple_receip | REC-8847 | CloudHost Services | DataFlow Analyt | $406.00 | March 22, 2024 |
| ✅ | invoice_003_email_style | DP-2024-156 | DesignPro Studio | RetailMax | $15,000.00 | April 5, 2024 |
| ✅ | invoice_004_european_form | MS-2024-0892 | Müller & Schmidt Gmb | Nordic Tech AS | $11,959.50 | 2024-06-18 |
| ✅ | invoice_005_handwritten_s | PP-5521 | Pete's Plumbing Serv | Mr. James Wilso | $355.00 | 2024-05-10 |
| ✅ | invoice_006_minimal | QK-99012 | QuickShip Logistics | FreshMart Groce | $65.00 | 2024-07-28 |



---

### 💰 Summary

- **Invoices processed:** 6
- **Successful extractions:** 6
- **Total value extracted:** $52,471.96

## 🎯 Summary

### What You Built

You built a document data extraction pipeline that uses four progressively more robust strategies — zero-shot, few-shot, JSON mode, and Pydantic validation — to extract structured invoice data from six wildly different document formats.

### Key Takeaways

1. **Zero-shot extraction** — works with no setup but produces inconsistent output formats
2. **Few-shot extraction** — providing examples dramatically improves output consistency
3. **JSON mode** — guarantees valid JSON responses, eliminating parsing failures
4. **Pydantic validation** — adds type safety and schema enforcement on top of LLM output
5. **Format resilience** — a well-defined schema lets the LLM extract the same structure from vastly different document formats